# 08 — Propensity Score Analysis

## Research question

Does exposure to a chronic health condition affect economic outcomes (placeholder specification — confirm treatment/outcome with supervisors and update `docs/decisions_log.md`).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_data
from src.cleaning import standardise_columns
from src.causal import (estimate_propensity_scores, nearest_neighbour_match,
                         check_covariate_balance)
from src.visualisation import plot_covariate_balance
from src.config import load_config

config = load_config()
df = standardise_columns(load_data('synthetic'))

treatment_col = config['variables']['health_exposure']
covariates = ['age']  # extend with sex/education once one-hot encoded appropriately

df['propensity_score'] = estimate_propensity_scores(df, treatment_col, covariates)
df[['person_id', 'year', treatment_col, 'age', 'propensity_score']].head()

## Covariate balance before matching

In [ ]:
balance_before = check_covariate_balance(df.dropna(subset=['propensity_score']), treatment_col, covariates)
balance_before

## Matching

In [ ]:
matched = nearest_neighbour_match(
    df.dropna(subset=['propensity_score']),
    propensity_col='propensity_score',
    treatment_col=treatment_col,
    caliper=config['propensity_score']['caliper'],
    caliper_scale=config['propensity_score'].get('caliper_scale', 'logit_sd'),
)
print('Matched pairs:', len(matched) // 2)

In [ ]:
balance_after = check_covariate_balance(matched, treatment_col, covariates)
fig, ax = plot_covariate_balance(
    balance_after, save_path='../outputs/figures/covariate_balance.png'
)

In [ ]:
# Log this output in the results register so it's traceable later.
from src.results_register import log_output

log_output(
    output_id='fig_covariate_balance_after_matching',
    source='notebooks/08_propensity_score.ipynb',
    research_question='RQ1 -- chronic condition and economic participation capacity',
    description='Standardised mean differences for covariates after nearest-neighbour matching',
    file_path='outputs/figures/covariate_balance.png',
    status='draft',
)
